In [1]:
from collections.abc import Callable, Iterable
import math
from typing import List, Optional, Union

import torch
from torch.optim import (
    Optimizer,
)
from torch.optim.optimizer import ParamsT

In [2]:
class SGD(Optimizer):
    def __init__(
        self,
        params: ParamsT,
        lr: Union[float, torch.Tensor] = 1e-3,
    ):
        if lr<0:
            raise ValueError(f"Invalid learning rate: {lr}".format(lr))
        defaults = {"lr": lr}
        super().__init__(params, defaults)
    
    def step(
        self,
        closure: Optional[Callable]=None
    ):
        # closure: function that re-evalutes model and returns loss
        loss = None if closure is None else closure()
        
        for group in self.param_groups:
            lr = group["lr"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                    
                state = self.state[p] # get state associated with p
                t = state.get("t", 0) # get iteration number
                grad = p.grad.data # get gradient of loss wrt p
                
                # Perform Update
                p.data -= (lr/math.sqrt(t+1)) * grad # lr/sqrt(t+1) is lr decay (not default in sgd)
                state["t"] = t+1
                
        return loss

In [19]:
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=1)

In [20]:
for t in range(10):
    opt.zero_grad() # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    loss.backward() # Run backward pass, which computes gradients.
    opt.step() # Run optimizer step
    print("loss {:.3f} lr {:.3f} t {}".format(
        loss.cpu().item(),
        opt.param_groups[0]["lr"],
        opt.state[
            opt.param_groups[0]["params"][0]
        ].get("t", 0),
    ))

loss 20.199 lr 1.000 t 1
loss 19.399 lr 1.000 t 2
loss 18.855 lr 1.000 t 3
loss 18.422 lr 1.000 t 4
loss 18.055 lr 1.000 t 5
loss 17.734 lr 1.000 t 6
loss 17.445 lr 1.000 t 7
loss 17.182 lr 1.000 t 8
loss 16.940 lr 1.000 t 9
loss 16.715 lr 1.000 t 10
